# NIH ChestX-ray14 — Multi-label Classification (NOT single binary)
This notebook trains a **multi-label** model on `Finding Labels` (excluding `No Finding`) and performs:

- Multi-label training (sigmoid outputs for each label)
- **Per-label thresholds** (selected on validation via **F1-optimal**)
- Full evaluation on test:
  - Per-label AUC / PR-AUC
  - Per-label Precision / Recall / F1 using thresholds
  - Micro/Macro averages
- Confusion matrix plots in the classic **blue** style (cmap="Blues")

In [ ]:
import os, glob, random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve,
    classification_report,
    multilabel_confusion_matrix,
    confusion_matrix,
    f1_score, precision_score, recall_score,
)

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

## 1) Load metadata CSV + NIH official splits
Adjust `dataset_path` if needed.

In [ ]:
# ===== Paths (adjust if needed) =====
dataset_path = "/path/to/local/resource"

csv_path      = os.path.join(dataset_path, "Data_Entry_2017.csv")
trainval_txt  = os.path.join(dataset_path, "train_val_list.txt")
test_txt      = os.path.join(dataset_path, "test_list.txt")

assert os.path.exists(csv_path), f"CSV not found: {csv_path}"
assert os.path.exists(trainval_txt), f"train_val_list.txt not found: {trainval_txt}"
assert os.path.exists(test_txt), f"test_list.txt not found: {test_txt}"

df = pd.read_csv(csv_path)
df["Image Index"] = df["Image Index"].astype(str).str.strip()
df["Finding Labels"] = df["Finding Labels"].astype(str).str.strip()

with open(trainval_txt, "r") as f:
    trainval_list = set([line.strip() for line in f if line.strip()])
with open(test_txt, "r") as f:
    test_list = set([line.strip() for line in f if line.strip()])

print("CSV rows:", len(df))
print("Train/Val list:", len(trainval_list))
print("Test list:", len(test_list))

## 2) Map Image Index → PNG path
The NIH dataset stores images across folders like `images_001/images/*.png`, etc.

In [ ]:
# ===== Index all PNGs =====
image_dirs = []
for d in os.listdir(dataset_path):
    if d.startswith("images_"):
        inner = os.path.join(dataset_path, d, "images")
        if os.path.isdir(inner):
            image_dirs.append(inner)

print("Image folders found:", len(image_dirs))
assert len(image_dirs) > 0, "No images_*/images folders found under dataset_path"

image_path_map = {}
for img_dir in image_dirs:
    for p in glob.glob(os.path.join(img_dir, "*.png")):
        image_path_map[os.path.basename(p)] = p

df["Path"] = df["Image Index"].map(image_path_map)
missing = df["Path"].isna().sum()
print("Missing images:", missing)
assert missing == 0, "Some images are missing from path mapping!"

## 3) Build multi-label targets (multi-hot)
- We **exclude** `No Finding` from the output labels.
- We drop contradictory rows where `No Finding` appears together with other labels (rare but cleaner).

In [ ]:
# ===== Build label vocabulary (excluding "No Finding") =====
all_labels = set()
for s in df["Finding Labels"].values:
    parts = [p.strip() for p in str(s).split("|")]
    for p in parts:
        if p and p != "No Finding":
            all_labels.add(p)

labels = sorted(list(all_labels))
K = len(labels)
label2idx = {l:i for i,l in enumerate(labels)}

print("Number of labels (excluding No Finding):", K)
print("Labels:", labels)

def encode_multihot(label_str: str) -> np.ndarray:
    y = np.zeros((K,), dtype=np.float32)
    parts = [p.strip() for p in str(label_str).split("|")]
    for p in parts:
        if p in label2idx:
            y[label2idx[p]] = 1.0
    return y

Y = np.stack(df["Finding Labels"].apply(encode_multihot).values, axis=0)

# Identify "No Finding" rows
is_no_finding = (df["Finding Labels"] == "No Finding").values
# Contradictions: No Finding + any other label
contrad = is_no_finding & (Y.sum(axis=1) > 0)

print("Contradictions (No Finding + other labels):", int(contrad.sum()))

df_ml = df.copy()
df_ml["Y"] = list(Y)

# Drop contradictions
df_ml = df_ml[~contrad].copy()
print("Rows after dropping contradictions:", len(df_ml))

## 4) Apply NIH split and create a validation set
We use the official NIH train/val list and test list.  
Then we split trainval into train/val (90/10) randomly.

In [ ]:
# ===== Apply NIH official split =====
df_trainval = df_ml[df_ml["Image Index"].isin(trainval_list)].copy()
df_test     = df_ml[df_ml["Image Index"].isin(test_list)].copy()

assert df_trainval["Image Index"].isin(test_list).sum() == 0, "Leakage: trainval contains test!"

print("Train/Val:", len(df_trainval), "| Test:", len(df_test))

# Shuffle trainval and make val split
df_trainval = df_trainval.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

val_frac = 0.10
n_val = int(len(df_trainval) * val_frac)
df_val = df_trainval.iloc[:n_val].copy()
df_train = df_trainval.iloc[n_val:].copy()

print("Train:", len(df_train), "| Val:", len(df_val))

# Quick label prevalence stats (train)
Y_train = np.stack(df_train["Y"].values, axis=0)
pos_counts = Y_train.sum(axis=0).astype(int)
prev = pos_counts / len(df_train)

stats = pd.DataFrame({"label": labels, "train_pos": pos_counts, "train_prev": prev})
stats = stats.sort_values("train_pos", ascending=False)
stats.head(10)

## 5) tf.data pipeline

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

AUTOTUNE = tf.data.AUTOTUNE

def parse_png(path, y):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    y = tf.cast(y, tf.float32)
    return img, y

def make_ds(df_in, training=False):
    paths = df_in["Path"].values.astype(str)
    Y = np.stack(df_in["Y"].values, axis=0).astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((paths, Y))
    if training:
        ds = ds.shuffle(buffer_size=min(len(df_in), 20000), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(parse_png, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(df_train, training=True)
val_ds   = make_ds(df_val, training=False)
test_ds  = make_ds(df_test, training=False)

print("✅ train/val/test datasets ready")

## 6) Handle imbalance: Weighted BCE per label
This is the multi-label equivalent of class weights.
We compute `pos_weight[l] = (neg+1)/(pos+1)` on TRAIN only and use it inside the loss.

In [ ]:
# ===== Compute per-label pos_weight from TRAIN =====
Y_train = np.stack(df_train["Y"].values, axis=0).astype(np.float32)
pos = Y_train.sum(axis=0)
neg = Y_train.shape[0] - pos

pos_weight = (neg + 1.0) / (pos + 1.0)

# Optional: clip extremely large weights for very rare labels
POS_WEIGHT_CLIP_MAX = 50.0
pos_weight = np.clip(pos_weight, 1.0, POS_WEIGHT_CLIP_MAX)

print("pos_weight (first 10):", np.round(pos_weight[:10], 2))

def weighted_bce(pos_weights):
    pos_weights = tf.constant(pos_weights, dtype=tf.float32)

    def loss(y_true, y_pred):
        eps = 1e-7
        y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)

        bce = -(y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        w = 1.0 + y_true * (pos_weights - 1.0)  # apply weights only to positives
        return tf.reduce_mean(bce * w)
    return loss

## 7) Model (baseline)
EfficientNetB0 pretrained on ImageNet → frozen feature extractor + small head.

In [ ]:
base = tf.keras.applications.EfficientNetB0(
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    weights="imagenet"
)
base.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation="relu")(x)
x = tf.keras.layers.Dropout(0.4)(x)
outputs = tf.keras.layers.Dense(K, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=weighted_bce(pos_weight),
    metrics=[tf.keras.metrics.AUC(name="auc", multi_label=True, num_labels=K)]
)

model.summary()

## 8) Train

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3, restore_best_weights=True)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=callbacks
)

## 9) Predict on VAL and TEST

In [ ]:
def collect(ds):
    y_true = []
    y_pred = []
    for xb, yb in ds:
        y_true.append(yb.numpy())
        y_pred.append(model.predict(xb, verbose=0))
    return np.concatenate(y_true, axis=0), np.concatenate(y_pred, axis=0)

y_val, p_val = collect(val_ds)
y_test, p_test = collect(test_ds)

print("VAL:", y_val.shape, p_val.shape)
print("TEST:", y_test.shape, p_test.shape)

## 10) Multi-label evaluation (AUC & PR-AUC)
We compute per-label ROC-AUC and PR-AUC, plus macro/micro averages.
Some labels may have no positives in the test split → we skip AUC for those safely.

In [ ]:
def safe_auc(y_true, y_score):
    # ROC-AUC requires both classes present
    if np.unique(y_true).size < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)

def safe_ap(y_true, y_score):
    # AP can work with no positives but it's not meaningful; we return NaN if no positives
    if y_true.sum() == 0:
        return np.nan
    return average_precision_score(y_true, y_score)

per_label = []
for l in range(K):
    yt = y_test[:, l]
    pt = p_test[:, l]
    per_label.append({
        "label": labels[l],
        "test_pos": int(yt.sum()),
        "test_prev": float(yt.mean()),
        "auc": safe_auc(yt, pt),
        "pr_auc": safe_ap(yt, pt),
    })

df_scores = pd.DataFrame(per_label)
df_scores_sorted = df_scores.sort_values("pr_auc", ascending=False)

macro_auc = np.nanmean(df_scores["auc"].values)
macro_pr  = np.nanmean(df_scores["pr_auc"].values)

# micro averages (flatten across labels)
micro_auc = safe_auc(y_test.ravel(), p_test.ravel())
micro_pr  = safe_ap(y_test.ravel(), p_test.ravel())

print("Macro ROC-AUC:", macro_auc)
print("Macro PR-AUC :", macro_pr)
print("Micro ROC-AUC:", micro_auc)
print("Micro PR-AUC :", micro_pr)

df_scores_sorted.head(10)

## 11) Choose per-label thresholds on VAL (F1-optimal)
This gives you a threshold **for each label**, not a single binary decision.

In [ ]:
# ===== Threshold selection on VAL: maximize F1 per label =====
thresholds = np.full((K,), 0.5, dtype=np.float32)

MIN_POS_FOR_THRESHOLD = 5  # if too rare in val, keep 0.5

for l in range(K):
    yt = y_val[:, l]
    pt = p_val[:, l]

    if yt.sum() < MIN_POS_FOR_THRESHOLD or np.unique(yt).size < 2:
        thresholds[l] = 0.5
        continue

    prec, rec, th = precision_recall_curve(yt, pt)
    # th has length len(prec)-1
    f1 = 2 * prec * rec / (prec + rec + 1e-9)

    best = np.argmax(f1[:-1])
    thresholds[l] = th[best]

# Optional: avoid extremely tiny thresholds that cause "everything positive"
THRESH_MIN = 0.05
THRESH_MAX = 0.95
thresholds = np.clip(thresholds, THRESH_MIN, THRESH_MAX)

# Show a few thresholds (top labels by test PR-AUC)
print("Example thresholds (top 10 labels by test PR-AUC):")
for name in df_scores_sorted["label"].head(10).tolist():
    l = label2idx[name]
    print(f"  {name:<18} t={thresholds[l]:.3f}")

## 12) Apply thresholds → multi-label predictions and full evaluation
We compute:
- Per-label precision/recall/F1
- Micro & Macro F1
- Multilabel confusion matrices (2x2 for each label)

In [ ]:
# Apply thresholds to TEST predictions
y_pred = (p_test >= thresholds[None, :]).astype(int)

# Micro/Macro F1
micro_f1 = f1_score(y_test, y_pred, average="micro", zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
micro_prec = precision_score(y_test, y_pred, average="micro", zero_division=0)
micro_rec  = recall_score(y_test, y_pred, average="micro", zero_division=0)

print("Micro Precision:", micro_prec)
print("Micro Recall   :", micro_rec)
print("Micro F1       :", micro_f1)
print("Macro F1       :", macro_f1)

# Per-label P/R/F1
per_label_prf = []
cms = multilabel_confusion_matrix(y_test, y_pred)  # shape (K, 2, 2) with [[tn, fp],[fn,tp]]

for l in range(K):
    tn, fp, fn, tp = cms[l].ravel()
    yt = y_test[:, l]
    per_label_prf.append({
        "label": labels[l],
        "threshold": float(thresholds[l]),
        "test_pos": int(yt.sum()),
        "auc": float(df_scores.loc[df_scores["label"] == labels[l], "auc"].values[0]),
        "pr_auc": float(df_scores.loc[df_scores["label"] == labels[l], "pr_auc"].values[0]),
        "precision": tp / (tp + fp + 1e-9),
        "recall": tp / (tp + fn + 1e-9),
        "f1": 2 * tp / (2*tp + fp + fn + 1e-9),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    })

df_eval = pd.DataFrame(per_label_prf).sort_values("pr_auc", ascending=False)

# Save full table
df_eval.to_csv("cxr_multilabel_eval.csv", index=False)
print("Saved: cxr_multilabel_eval.csv")

df_eval.head(10)

## 13) Classification report (optional, can be long)

In [ ]:
# WARNING: this can be long if K is large, but it's useful for debugging
print(classification_report(y_test, y_pred, target_names=labels, zero_division=0))

## 14) Confusion matrix plots (classic blue style)
Because this is multi-label, we provide:
1) **Micro confusion matrix** (flattened across all labels)
2) Per-label confusion matrices for top-N labels (by PR-AUC)

In [ ]:
# 1) Micro confusion matrix (flatten all label decisions)
y_true_flat = y_test.ravel()
y_pred_flat = y_pred.ravel()

cm_micro = confusion_matrix(y_true_flat, y_pred_flat)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_micro)
plt.figure()
disp.plot(values_format="d", cmap="Blues")
plt.title("Micro Confusion Matrix (All labels flattened)")
plt.tight_layout()
plt.savefig("cxr_confusion_matrix_micro.png", dpi=200)
plt.close()
print("Saved: cxr_confusion_matrix_micro.png")

# 2) Per-label confusion matrices for top-N labels
TOP_N = 6
top_labels = df_eval["label"].head(TOP_N).tolist()

for name in top_labels:
    l = label2idx[name]
    tn, fp, fn, tp = cms[l].ravel()
    cm = np.array([[tn, fp],
                   [fn, tp]])

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["0", "1"])
    plt.figure()
    disp.plot(values_format="d", cmap="Blues")
    plt.title(f"Confusion Matrix — {name} (threshold={thresholds[l]:.3f})")
    plt.tight_layout()
    fname = f"cxr_confusion_matrix_{name.replace(' ', '_')}.png"
    plt.savefig(fname, dpi=200)
    plt.close()

print("Saved per-label confusion matrices for:", top_labels)

## 15) (Optional) Save PR curves for top labels
Great for posters.

In [ ]:
from sklearn.metrics import precision_recall_curve

TOP_N_PR = 4
top_labels = df_scores_sorted["label"].head(TOP_N_PR).tolist()

for name in top_labels:
    l = label2idx[name]
    yt = y_test[:, l]
    pt = p_test[:, l]

    if yt.sum() < 5:
        continue

    prec, rec, _ = precision_recall_curve(yt, pt)
    plt.figure()
    plt.plot(rec, prec)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"PR Curve — {name}")
    plt.tight_layout()
    fname = f"cxr_pr_{name.replace(' ', '_')}.png"
    plt.savefig(fname, dpi=200)
    plt.close()

print("Saved PR curves for:", top_labels)